## Baseball Pitch and Zone Prediction Project

Our solution is architected in three distinct modules that run sequentially:
1.  **The Deep Eye:** A 3D Convolutional Neural Network (CNN) that watches the video to extract visual features.
2.  **The Physics Simulator:** A Neural Network that learns to solve kinematic equations to predict the ball's trajectory.
3.  **The Grandmaster Fusion:** A CatBoost model that takes the outputs of the first two modules (Vision + Physics) and makes the final decision.

### 1. The Data Ingestion Layer (PitchVideoDataset)
This class converts raw .mp4 video files into mathematical Tensors $(C, T, H, W)$ that a neural network can digest.

* **Low-Level Operation:**
    * **Frame Sampling:** It does not read every frame (which would be redundant). Instead, it uses np.linspace to uniformly sample exactly 16 frames from the entire clip. This captures the temporal structure (start, middle, end) regardless of the video's original frame rate.
    * **Tensor Transformation:**
        * **Resizing:** Every frame is squashed to $112 \times 112$ pixels.
        * **Color Channel Flip:** OpenCV reads in BGR; this manually converts it to RGB.
        * **Normalization:** It divides pixel intensity by 255.0 (scaling 0–255 to 0.0–1.0).
        * **Permutation:** It transposes the dimensions from (Time, Height, Width, Channels) to (Channels, Time, Height, Width) because PyTorch 3D convolutions expect the Channel dimension first.

### 2. Module A: The Deep Eye (3D Computer Vision)
This is the Vision component. It uses a ResNet-3D (R3D-18) architecture.

* **Architecture Details:**
    * **Backbone:** r3d_18 pre-trained on Kinetics-400.
        * Standard 2D CNNs slide a flat filter over an image. This model slides a 3D volumetric filter (a cube) over the video block. This allows it to learn features that exist in *time* (like the downward snap of a curveball) rather than just static shapes.
    * **The Model Embedding Extraction (model.fc = nn.Identity()):** The original model was built to classify actions. This notebook rips off that final classification layer. Instead of a class prediction, it intercepts the raw 512-dimensional embedding vector from the layer just before the end.
    * **Output:** For every video, it outputs a vector of 512 floating-point numbers. These numbers represent the identiy of the pitch's motion.

### 3. Module B: The Physics Simulator (Differentiable Physics)
This module is a Gray Box model consisting of part Neural Network (Black Box), part High School Physics (White Box).

* **The Brain (nn.Sequential):**
    * **Input:** 5 physics variables (Speed, Spin, Release Position).
    * **Hidden Layers:** Two dense layers (512 $\to$ 256) with Batch Normalization and SiLU activation.
    * **Output:** It predicts 3 Aerodynamic Coefficients:
        * ax: Horizontal acceleration (Magnus force left/right).
        * az: Vertical acceleration (Magnus force up/down).
        * vz_corr: Correction factor for vertical velocity.

* **The Physics Layer:**
    * Instead of letting the neural network guess the final position directly, the code forces the output through Newton's Kinematic Equation:
        $$Position_{final} = Position_{start} + (Velocity \times Time) + (0.5 \times Acceleration \times Time^2)$$
    * The model is now forced to find the specific aerodynamic forces that explain the ball's path.

### 4. Module C: The Grandmaster Fusion (CatBoost)
This module serves as the final decision maker that aggregates all the prior information.

* **Inputs:**
    * **Raw Metadata:** Release speed, spin rate, extension.
    * **Physics Predictions:** The sim_x and sim_z coordinates calculated by Module B.
    * **Visual Embeddings:** The 512 Deep Eye features extracted by Module A.
* **The Regressor:**
    * It uses MAE (Mean Absolute Error) loss, which is less sensitive to outliers than RMSE.
    * It trains for 25,000 iterations with a small learning rate (0.01), enabling it to learn extremely subtle interactions between the video features and the physics data.

### 5. The Final Logic (Geometric Umpire)
This final step is a deterministic algorithm (assign_zone).

* It takes the $(x, z)$ coordinates predicted by CatBoost.
* It draws a virtual box around the strike zone (17 inches wide).
* **Rule:** If the coordinates are inside the box $\to$ Strike. If outside $\to$ Ball.
* It then mathematically slices that box into a $3 \times 3$ grid to assign the exact Zone ID (1–9).

### Summary of Data Flow
1.  **Video File** $\xrightarrow{\text{PitchVideoDataset}}$ **Tensor** $(3, 16, 112, 112)$
2.  **Tensor** $\xrightarrow{\text{Deep Eye}}$ **Embedding** $(512,)$
3.  **Statcast Data** $\xrightarrow{\text{Physics Engine}}$ **Simulated Coords** $(x_{sim}, z_{sim})$
4.  **Embedding + Simulation** $\xrightarrow{\text{CatBoost}}$ **Final Coords** $(x_{final}, z_{final})$
5.  **Final Coords** $\xrightarrow{\text{Geometric Logic}}$ **Prediction** (e.g., Strike, Zone 5)

Note: This Project was completed by Josue Flores and Akshat Singh.

In [ ]:
# CELL 1: Imports and Device
import torch
import torch.nn as nn
import torch.optim as optim
from torchdiffeq import odeint
import numpy as np
import pandas as pd
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


This cell defines the Differential Equations of Motion for the baseball.

* **System of ODEs:** It implements the standard kinematic laws of physics:
    * **Position Change:** Determined by Velocity ($dx/dt$ = $v_x$).
    * **Velocity Change:** Determined by Acceleration ($dv/dt = a$).
* **Hybrid Physics:**
    * **Hard-Coded:** It manually subtracts Gravity (32.174 ft/s²) from the vertical acceleration, ensuring the model obeys Newton's laws.
    * **Learned:** It uses a sub-model (self.accel_model) to predict the Aerodynamic Forces ($a_x, a_z$) based on the pitch's features.
* **Purpose:** This module is designed to be passed into an ODE Solver (like torchdiffeq.odeint). The solver will call this function repeatedly to integrate the trajectory from the pitcher's hand to the plate.

In [ ]:
# CELL 2: ODE Dynamics Function

class PitchDynamics(nn.Module):
    """
    dx/dt = vx
    dz/dt = vz
    dvx/dt = ax
    dvz/dt = az - g
    """

    def __init__(self, accel_model):
        super().__init__()
        self.accel_model = accel_model
        self.g = 32.174  # ft/s^2 downward

    def forward(self, t, state):
        # state = [x, z, vx, vz]
        x, z, vx, vz = torch.split(state, 1, dim=1)

        # Predict aerodynamic accel (ax, az) from features
        ax_az = self.accel_model(self.features)  # [B, 2]
        ax = ax_az[:, 0:1]
        az = ax_az[:, 1:2]

        dxdt = vx
        dzdt = vz
        dvxdt = ax
        dvzdt = az - self.g

        return torch.cat([dxdt, dzdt, dvxdt, dvzdt], dim=1)

This cell defines the Brain of the physics simulator. It is a simple Feed-Forward Neural Network (Multi-Layer Perceptron) that learns the complex relationship between a pitch's spin/speed and how much it curves.

* **The Input:** It takes a vector of pitch features (input_dim), which likely corresponds to things like release_speed, spin_rate, and release_pos.
* **The Architecture:** It is a standard deep network with two hidden layers:
    * **Layer 1:** Expands the input to 256 neurons (learning complex interactions).
    * **Layer 2:** Compresses features to 128 neurons.
    * **Activation:** Uses ReLU to capture non-linear physics (how drag increases quadratically with speed).
* **The Output$:$** The final layer outputs exactly 2 numbers:
    1.  $a_x$: The horizontal acceleration (how much the ball breaks left/right).
    2.  $a_z$: The vertical aerodynamic lift (how much the ball rises or drops due to spin).

In [ ]:
# CELL 3: Neural Acceleration Model (MLP)

class AccelMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 2)   # ax, az
        )
    def forward(self, x):
        return self.net(x)

This cell connects the Neural Network brain (AccelMLP) with the Physics equations (PitchDynamics) and executes the flight simulation.

* **Initialization (__init__):**
    * It instantiates the AccelMLP. This serves as the learnable component that will figure out the aerodynamic forces.


* **Setting the Scene (forward)$:$**
    * **State Packing:** It takes the initial conditions, release position $(x_0, z_0)$ and release velocity $(v_{x0}, v_{z0})$, and packs them into a single State Tensor. This represents the ball at the exact moment it leaves the pitcher's hand (t=0).
    * **Context Passing:** It injects the features (spin, speed, etc.) into the dynamics object so the differential equations know *which* specific pitch is flying through the air.

* **The Simulation (odeint):**
    * **Integration Time:** It calculates how long the simulation needs to run. Note the simplification: it uses t_plate.mean().item(), meaning it integrates the entire batch for the *average* flight time of that batch.
    * **The Solver:** It calls torchdiffeq.odeint. This is the "black box" numerical integrator.
        * **Method:** It uses rk4, a standard algorithm for solving differential equations step-by-step.
        * **Action:** It effectively "throws" the ball, calculating its position and velocity at every tiny time step based on the physics defined in Cell 2.

* **The Output:**
    * It grabs the final state (sol[-1]) at the end of the flight time.
    * It extracts just the final position $(pred_x, pred_z)$, which corresponds to where the ball crosses home plate. This is the value that will be compared against the ground truth during training.

In [ ]:
# CELL 4: NeuralODE Wrapper

class PitchNeuralODE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.accel_model = AccelMLP(input_dim)

    def forward(self, features, t_plate, x0, z0, vx0, vz0):
        # dynamics object that uses accel_model
        dynamics = PitchDynamics(self.accel_model)
        dynamics.features = features

        # initial state shape [B,4]
        state0 = torch.cat([x0, z0, vx0, vz0], dim=1)

        # t must be 1-D: pick a representative integration time (mean across batch)
        # torchdiffeq expects t: [T], so [0, t_final]
        t_final = t_plate.mean().item()
        t = torch.tensor([0.0, t_final], dtype=torch.float32).to(DEVICE)

        # ODE solve: output shape [2, B, 4]
        sol = odeint(dynamics, state0, t, method='rk4')

        # final state at t_final
        final = sol[-1]       # [B,4]
        pred_x = final[:, 0:1]
        pred_z = final[:, 1:2]
        return pred_x, pred_z

This cell is responsible for gathering the massive dataset required to teach the Physics Engine how baseballs move in the real world.

**The Source**: It uses pybaseball.statcast to scrape official MLB data from the 2021, 2022, and 2023 seasons.

**Caching Mechanism**: To save time, it checks if physics_calibration_3yrs.csv already exists. If so, it loads it instantly. If not, it performs the long download process.

**Data Cleaning**: It selects only the 13 columns critical for physics (Speed, Spin, Release Point, Break, Plate Location) and drops any rows with missing values to ensure the training data is clean.

In [ ]:
# Cell 5a : Acquire 3 YEARS OF Statcast-trained physics dataset

# Install pybaseball if needed
try:
    from pybaseball import statcast
except ImportError:
    os.system('pip install pybaseball')
    from pybaseball import statcast

def fetch_grandmaster_data():
    save_file = "physics_calibration_3yrs.csv"
    if os.path.exists(save_file):
        print(" Loading cached 3-year physics data...")
        return pd.read_csv(save_file)

    print(" Downloading 3 Years of Physics Data (2021-2023)...")
    dfs = []
    for yr in [2021, 2022, 2023]:
        try:
            print(f"   Fetching {yr}...")
            d = statcast(start_dt=f'{yr}-04-01', end_dt=f'{yr}-10-01')
            dfs.append(d)
        except: pass

    if not dfs:
        print(" Download failed. Using short fallback.")
        return statcast(start_dt='2024-04-01', end_dt='2024-06-01')

    df = pd.concat(dfs)
    cols = ['release_speed', 'release_spin_rate', 'release_extension',
            'release_pos_x', 'release_pos_z', 'pfx_x', 'pfx_z',
            'sz_top', 'sz_bot', 'stand', 'p_throws', 'plate_x', 'plate_z']

    df = df[cols].dropna().reset_index(drop=True)
    df.to_csv(save_file, index=False)
    return df

df_ext = fetch_grandmaster_data()

This cell performs the Data Preprocessing and Tensor Transformation required to feed the Statcast data into the Neural Physics Engine. It converts raw baseball statistics into the specific mathematical inputs the differential equations need.

* **Feature Selection:** It isolates the 5 core variables that drive the physics simulation: Release Speed, Spin Rate, Release Position (X/Z), and Handedness (p_throws).
* **Physics Derivations:** It calculates necessary kinematic variables that aren't explicitly in the CSV:
    * **$v_{fps}$:** Converts speed from MPH to feet per second.
    * **$t_{plate}$:** Calculates the time of flight to the plate based on release extension.
    * **$v_{z0}$:** Estimates the initial *vertical* velocity vector by calculating the launch angle required to reach the bottom of the strike zone.
* **Normalization (QuantileTransformer):** It transforms the input features into a standard Normal distribution. This is critical for the Neural Network to learn efficiently.
* **Tensor Creation:** It converts all numpy arrays into GPU-ready PyTorch Tensors ($X_t$, $t_{plate}$, $x_0$, $z_0$, $v_{z0}$, etc.) and reshapes them to (N, 1) to be compatible with batch processing in the training loop.

In [ ]:
# CELL 5b (FIXED): Load and prepare Statcast-trained physics dataset

physics_cols = [
    'release_speed',
    'release_spin_rate',
    'release_pos_x',
    'release_pos_z',
    'p_throws'
]

df = pd.read_csv("physics_calibration_3yrs.csv").dropna().reset_index(drop=True)

# Map categorical columns BEFORE scaling
mapping = {'L': 0, 'R': 1}
if df['p_throws'].dtype == object:
    df['p_throws'] = df['p_throws'].map(mapping)
if 'stand' in df.columns and df['stand'].dtype == object:
    df['stand'] = df['stand'].map(mapping)

# Compute v_fps
v_fps = (df['release_speed'] * 1.467).replace(0, 100)  # mph -> ft/s

# Compute ToF to plate
t_plate_np = ((60.5 - df['release_extension']) / v_fps).values

# Compute approximate vertical velocity vz0
# Need the batter strike zone bottom to estimate vertical angle
if 'sz_bot' in df.columns:
    release_angle = np.arctan((df['release_pos_z'] - df['sz_bot']) / 60.5)
else:
    # fallback: small vertical angle if unavailable
    release_angle = np.zeros(len(df))

vz0_np = v_fps.values * np.sin(release_angle)

# Horizontal velocity unavailable → assume 0 for now
vx0_np = np.zeros_like(vz0_np)

# Scale physics inputs
from sklearn.preprocessing import QuantileTransformer
scaler = QuantileTransformer(output_distribution='normal')
X_phys = scaler.fit_transform(df[physics_cols].values)

# Convert to tensors
X_t = torch.tensor(X_phys, dtype=torch.float32).to(DEVICE)

t_plate = torch.tensor(t_plate_np, dtype=torch.float32).to(DEVICE).unsqueeze(1)
x0 = torch.tensor(df['release_pos_x'].values, dtype=torch.float32).to(DEVICE).unsqueeze(1)
z0 = torch.tensor(df['release_pos_z'].values, dtype=torch.float32).to(DEVICE).unsqueeze(1)

vz0 = torch.tensor(vz0_np, dtype=torch.float32).to(DEVICE).unsqueeze(1)
vx0 = torch.tensor(vx0_np, dtype=torch.float32).to(DEVICE).unsqueeze(1)

y_x = torch.tensor(df['plate_x'].values, dtype=torch.float32).to(DEVICE).unsqueeze(1)
y_z = torch.tensor(df['plate_z'].values, dtype=torch.float32).to(DEVICE).unsqueeze(1)

print("Prepared:")
print("X_t:", X_t.shape)
print("t_plate:", t_plate.shape)
print("x0:", x0.shape)
print("vz0:", vz0.shape)
print("Targets:", y_x.shape, y_z.shape)

Prepared:
X_t: torch.Size([697972, 5])
t_plate: torch.Size([697972, 1])
x0: torch.Size([697972, 1])
vz0: torch.Size([697972, 1])
Targets: torch.Size([697972, 1]) torch.Size([697972, 1])


This cell executes the Training Loop for the Neural Physics Engine.

**Optimizer:** Uses AdamW with a learning rate of 0.001.

**Loss Function:** Uses nn.HuberLoss. This is crucial because baseball tracking data can have outliers (measurement errors). Huber loss is less sensitive to these outliers than Mean Squared Error (MSE), preventing the model from exploding due to a single bad data point.

**The Loop (15 Epochs)**:

Shuffling: It randomizes the order of pitches every epoch to prevent the model from memorizing the sequence.

Batching: It processes 4,096 pitches at a time.

Forward Pass: It runs the simulation (model(...)) to predict where the ball should land based on the current understanding of physics.

Backward Pass (Backpropagation): It calculates the error (Loss) between the simulated landing spot (pred_x, pred_z) and the actual landing spot (yxb, yzb). It then updates the neural network weights (optimizer.step()) to reduce this error for the next batch.

In [ ]:
# CELL 6: Train Neural ODE

model = PitchNeuralODE(input_dim=len(physics_cols)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
crit = nn.HuberLoss()

batch_size = 4096
num_epochs = 15

N = X_t.shape[0]
idx = np.arange(N)

for epoch in range(num_epochs):
    np.random.shuffle(idx)
    total_loss = 0

    for i in range(0, N, batch_size):
        batch = idx[i:i+batch_size]

        Xb  = X_t[batch]
        tb  = t_plate[batch]
        x0b = x0[batch]
        z0b = z0[batch]
        vx0b = vx0[batch]
        vz0b = vz0[batch]
        yxb = y_x[batch]
        yzb = y_z[batch]

        optimizer.zero_grad()
        pred_x, pred_z = model(Xb, tb, x0b, z0b, vx0b, vz0b)
        loss = crit(pred_x, yxb) + crit(pred_z, yzb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch", epoch+1, "Loss:", total_loss)

Epoch 1 Loss: 273.57123750448227
Epoch 2 Loss: 125.13504099845886
Epoch 3 Loss: 120.68090838193893
Epoch 4 Loss: 118.57698291540146
Epoch 5 Loss: 117.19749969244003
Epoch 6 Loss: 116.26385974884033
Epoch 7 Loss: 115.659716963768
Epoch 8 Loss: 115.17992234230042
Epoch 9 Loss: 114.80374205112457
Epoch 10 Loss: 114.5613494515419
Epoch 11 Loss: 114.35951739549637
Epoch 12 Loss: 114.20639824867249
Epoch 13 Loss: 114.0971947312355
Epoch 14 Loss: 113.91114920377731
Epoch 15 Loss: 113.86390978097916


This cell performs Feature Engineering via Inference. It applies the trained Physics Simulator (Neural ODE) to the actual Kaggle data to generate two features: sim_x_ode and sim_z_ode.

* **Preprocessing for Inference (preprocess_for_ode):**
    * **Data Sanitization:** Crucially, it cleans the time_to_plate values. It replaces NaNs or infinite values with 0.1 seconds and clips the time between 0.05 and 1.20 seconds. This prevents the ODE solver from crashing due to bad math (like dividing by zero speed).
    * **Standardization:** It uses the scaler (scaler.transform) that was fitted on the 3-year Statcast dataset. This ensures the competition data looks exactly like the training data to the neural network.
    * **Tensor Packing:** It converts the competition metadata into the exact tensor format expected by the model.

* **The Simulation Loop (gen_sim_features)$:$**
    * **Inference Mode:** It runs inside torch.no_grad() to disable gradient tracking, which speeds up calculation and saves memory since we aren't training anymore.
    * **Batching:** It processes the data in chunks of 4,096 rows to avoid overloading GPU memory.
    * **Execution:** It calls model(Xs, ts, x0s, z0s, vx0s, vz0s) for each batch. This triggers the ODE solver to integrate the flight path for every single pitch in the competition dataset.

* **Feature Injection:**
    * The output coordinates (preds_x, preds_z) are essentially the Physics Engine's Opinion on where the ball ended up.
    * These are added to the dataframes as new columns (sim_x_ode, sim_z_ode). These synthetic features will be extremely high-value inputs for the final CatBoost model because they encapsulate complex aerodynamics into two simple numbers.

In [ ]:
# CELL 7: Generate ODE features for train/test without retraining

mapping = {'L': 0, 'R': 1}

def preprocess_for_ode(df_input):
    df = df_input.copy()

    # Map categoricals
    if df['p_throws'].dtype == object:
        df['p_throws'] = df['p_throws'].map(mapping)
    if 'stand' in df.columns and df['stand'].dtype == object:
        df['stand'] = df['stand'].map(mapping)

    # Compute v_fps
    v_fps = (df['release_speed'] * 1.467).replace(0, 100)

    # Compute raw ToF
    t_plate_np = ((60.5 - df['release_extension']) / v_fps).values

    # Sanitize all invalid values BEFORE using in ODE
    t_plate_np = np.nan_to_num(t_plate_np, nan=0.1, posinf=0.1, neginf=0.1)
    t_plate_np = np.clip(t_plate_np, 0.05, 1.20)

    # Compute approximate vertical velocity vz0
    if 'sz_bot' in df.columns:
        release_angle = np.arctan((df['release_pos_z'] - df['sz_bot']) / 60.5)
    else:
        release_angle = np.zeros(len(df))

    vz0_np = v_fps.values * np.sin(release_angle)
    vx0_np = np.zeros_like(vz0_np)

    # Scale physics inputs
    X_phys = scaler.transform(df[physics_cols].values)

    # Build tensors
    Xb  = torch.tensor(X_phys, dtype=torch.float32).to(DEVICE)
    tb  = torch.tensor(t_plate_np, dtype=torch.float32).to(DEVICE).unsqueeze(1)
    x0b = torch.tensor(df['release_pos_x'].values, dtype=torch.float32).to(DEVICE).unsqueeze(1)
    z0b = torch.tensor(df['release_pos_z'].values, dtype=torch.float32).to(DEVICE).unsqueeze(1)
    vz0b = torch.tensor(vz0_np, dtype=torch.float32).to(DEVICE).unsqueeze(1)
    vx0b = torch.tensor(vx0_np, dtype=torch.float32).to(DEVICE).unsqueeze(1)

    return Xb, tb, x0b, z0b, vx0b, vz0b


def gen_sim_features(df_input):
    Xb, tb, x0b, z0b, vx0b, vz0b = preprocess_for_ode(df_input)

    preds_x = []
    preds_z = []

    with torch.no_grad():
        batch_size = 4096
        N = Xb.shape[0]

        for i in range(0, N, batch_size):
            Xs  = Xb[i:i+batch_size]
            ts  = tb[i:i+batch_size]
            x0s = x0b[i:i+batch_size]
            z0s = z0b[i:i+batch_size]
            vx0s = vx0b[i:i+batch_size]
            vz0s = vz0b[i:i+batch_size]

            # Forward() remains EXACTLY as during training
            pred_x, pred_z = model(Xs, ts, x0s, z0s, vx0s, vz0s)

            preds_x.append(pred_x.cpu().numpy())
            preds_z.append(pred_z.cpu().numpy())

    preds_x = np.concatenate(preds_x).ravel()
    preds_z = np.concatenate(preds_z).ravel()

    return preds_x, preds_z


# Load Kaggle files
train_df = pd.read_csv("/teamspace/studios/this_studio/baseball_kaggle_dataset_trimmed_only/data/train_ground_truth.csv")
test_df  = pd.read_csv("/teamspace/studios/this_studio/baseball_kaggle_dataset_trimmed_only/data/test_features.csv")

# Generate new ODE-based features
train_df['sim_x_ode'], train_df['sim_z_ode'] = gen_sim_features(train_df)
test_df['sim_x_ode'],  test_df['sim_z_ode']  = gen_sim_features(test_df)

In [ ]:
# CELL 8: Save Neural ODE Features to CSV

train_ode_path = "neural_ode_train_features.csv"
test_ode_path  = "neural_ode_test_features.csv"

train_df[['file_name', 'sim_x_ode', 'sim_z_ode']].to_csv(train_ode_path, index=False)
test_df[['file_name', 'sim_x_ode', 'sim_z_ode']].to_csv(test_ode_path, index=False)

print("Saved:")
print(train_ode_path)
print(test_ode_path)
print("Train ODE shape:", train_df[['file_name','sim_x_ode','sim_z_ode']].shape)
print("Test  ODE shape:", test_df[['file_name','sim_x_ode','sim_z_ode']].shape)

Saved:
neural_ode_train_features.csv
neural_ode_test_features.csv
Train ODE shape: (6000, 3)
Test  ODE shape: (4000, 3)


This cell aggregates all the intelligence generated by the previous modules into a single, massive dataset ready for the decision maker model (CatBoost).

* **Loading the Pieces:**
    * It loads the original competition data (train_ground_truth.csv, test_features.csv).
    * It loads the 512-dimensional Visual Features extracted by the Deep Eye 3D CNN (deep_video_features.csv).
    * It loads the Physics Predictions generated by the Neural ODE (neural_ode_features.csv).

* **The Merge (pd.merge):**
    * It joins all these datasets together using file_name as the unique key.
    * **Result:** Each row in your dataframe now contains:
        1.  **Metadata:** Pitcher throws Right, 85mph fastball.
        2.  **Vision:** The video looks like a high-spin curveball according to the embedding vector.
        3.  **Physics:** Newton's laws dictates it should land at x=0.5, z=2.1.

* **Cleanup:**
    * It fills any missing values with 0. This is a safety step to ensure the final model doesn't crash if a specific video file was corrupted or the tracker failed.

In [ ]:
# CELL 9: Merge Deep Video Embeddings + Neural ODE Features into train/test

# Paths — adjust if needed
BASE_DIR = "/teamspace/studios/this_studio/baseball_kaggle_dataset_trimmed_only/data/"
TRAIN_PATH = BASE_DIR + "train_ground_truth.csv"
TEST_PATH  = BASE_DIR + "test_features.csv"
DEEP_FEATS_PATH = BASE_DIR + "deep_video_features.csv"    # 512-d embeddings
ODE_TRAIN_PATH  = "neural_ode_train_features.csv"         # ODE features
ODE_TEST_PATH   = "neural_ode_test_features.csv"

# Load main data
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# Load deep video embeddings
deep_df = pd.read_csv(DEEP_FEATS_PATH)
emb_cols = [c for c in deep_df.columns if c.startswith("vid_emb_")]

# Merge DeepEye features
train = train.merge(deep_df, on="file_name", how="left")
test  = test.merge(deep_df, on="file_name", how="left")

# Merge Neural ODE features
train = train.merge(pd.read_csv(ODE_TRAIN_PATH), on="file_name", how="left")
test  = test.merge(pd.read_csv(ODE_TEST_PATH), on="file_name", how="left")

# Fill missing embedding or ODE values with 0
train[emb_cols] = train[emb_cols].fillna(0)
test[emb_cols]  = test[emb_cols].fillna(0)
train[['sim_x_ode','sim_z_ode']] = train[['sim_x_ode','sim_z_ode']].fillna(0)
test[['sim_x_ode','sim_z_ode']]  = test[['sim_x_ode','sim_z_ode']].fillna(0)

print("Merged train shape:", train.shape)
print("Merged test shape: ", test.shape)

train.head(3)

Merged train shape: (6000, 532)
Merged test shape:  (4000, 528)


,file_name,plate_x,plate_z,sz_top,sz_bot,release_speed,effective_speed,release_spin_rate,release_pos_x,release_pos_y,...,vid_emb_504,vid_emb_505,vid_emb_506,vid_emb_507,vid_emb_508,vid_emb_509,vid_emb_510,vid_emb_511,sim_x_ode,sim_z_ode
0,pitch1.mp4,1.24,3.32,3.52,1.57,93.7,92.5,2300.0,-1.35,54.79,...,0.712824,0.743943,0.534308,0.775923,0.482518,0.691621,0.572301,0.908545,-0.190525,3.272387
1,pitch3.mp4,-0.51,2.26,3.88,1.91,95.1,95.1,2412.0,2.14,54.06,...,0.616479,0.561221,0.554009,0.606378,0.454456,0.533403,0.476571,0.309303,0.208695,3.083644
2,pitch4.mp4,0.90,3.92,3.16,1.47,94.7,98.0,2471.0,2.37,52.39,...,0.723069,0.835482,0.490563,0.737871,0.391179,0.656291,0.621568,1.183055,0.233774,3.464612


In [ ]:
# CELL 10a: Ensure physics categorical columns are numeric before CatBoost

mapping = {'L': 0, 'R': 1}

for col in ['p_throws', 'stand']:
    if col in train.columns and train[col].dtype == object:
        train[col] = train[col].map(mapping)
    if col in test.columns and test[col].dtype == object:
        test[col] = test[col].map(mapping)

# After mapping, verify no strings remain
print("Unique train p_throws:", train['p_throws'].unique())
print("Unique test  p_throws:", test['p_throws'].unique())

Unique train p_throws: [1 0]
Unique test  p_throws: [1 0]


In [ ]:
# CELL 10b: Final feature list for CatBoost

# Core physics features to always include
phys_cols = [
    'release_speed', 'release_spin_rate',
    'release_pos_x', 'release_pos_z',
    'p_throws'
]

# Add Neural ODE features
ode_cols = ['sim_x_ode', 'sim_z_ode']

# Add DeepEye embedding columns
all_emb_cols = [c for c in train.columns if c.startswith("vid_emb_")]

# Final combined feature list
final_features = phys_cols + ode_cols + all_emb_cols

print("Number of final features:", len(final_features))


Number of final features: 519


In [ ]:
# CELL 11: Train CatBoost for plate_x

from catboost import CatBoostRegressor

cb_x = CatBoostRegressor(
    iterations=25000,
    learning_rate=0.01,
    depth=8,
    loss_function="MAE",
    task_type="GPU",
    verbose=5000
)

cb_x.fit(train[final_features], train['plate_x'])

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.6812271	total: 99.9ms	remaining: 41m 36s
5000:	learn: 0.0427318	total: 3m 10s	remaining: 12m 40s
10000:	learn: 0.0188014	total: 6m 22s	remaining: 9m 34s
15000:	learn: 0.0094765	total: 9m 36s	remaining: 6m 24s
20000:	learn: 0.0056455	total: 12m 51s	remaining: 3m 12s
24999:	learn: 0.0038972	total: 16m 5s	remaining: 0us


In [ ]:
# CELL 12: Train CatBoost for plate_z

cb_z = CatBoostRegressor(
    iterations=25000,
    learning_rate=0.01,
    depth=8,
    loss_function="MAE",
    task_type="GPU",
    verbose=5000
)

cb_z.fit(train[final_features], train['plate_z'])

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.7764295	total: 47ms	remaining: 19m 35s
5000:	learn: 0.0390960	total: 3m 8s	remaining: 12m 35s
10000:	learn: 0.0161070	total: 6m 20s	remaining: 9m 30s
15000:	learn: 0.0081379	total: 9m 34s	remaining: 6m 22s
20000:	learn: 0.0048365	total: 12m 48s	remaining: 3m 12s
24999:	learn: 0.0033834	total: 16m 3s	remaining: 0us


In [ ]:
# CELL 13: Predict test set plate_x and plate_z

test['guess_x'] = cb_x.predict(test[final_features])
test['guess_z'] = cb_z.predict(test[final_features])

test[['file_name', 'guess_x', 'guess_z']].head()

/tmp/ipykernel_10401/3056597842.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test['guess_x'] = cb_x.predict(test[final_features])
/tmp/ipykernel_10401/3056597842.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test['guess_z'] = cb_z.predict(test[final_features])


,file_name,guess_x,guess_z
0,pitch2.mp4,0.921804,3.666086
1,pitch6.mp4,1.182562,2.079888
2,pitch7.mp4,-0.535348,3.063244
3,pitch9.mp4,0.358600,2.947956
4,pitch10.mp4,0.582826,2.361661


This cell implements the Geometric Logic. It is a deterministic, rule-based function that converts the continuous (x, z) coordinates predicted by the CatBoost model into the discrete classification labels required for the competition submission.

* **Inputs:**
    * It takes the Predicted Location (guess_x, guess_z) generated by the Grandmaster Fusion model.
    * It takes the Strike Zone Boundaries (sz_top, sz_bot) specific to the batter in that clip.

* **The Strike Zone Definition (if abs(x) <= 0.83 ...):**
    * It defines a Strike using the official MLB rulebook width (17 inches / 2 = 0.708 ft) plus the radius of the baseball (~0.12 ft), totaling 0.83 ft.
    * If the ball's center is within this horizontal range and between the vertical top and bot, it is labeled a Strike.

* **Zone Assignment (1–9):**
    * If the pitch is a strike, the code mathematically slices the zone into a $3 \times 3$ grid.
    * It calculates the column (Left/Middle/Right) and row (Top/Middle/Bottom) to assign the standard Gameday Zones 1 through 9.

* **Zone Assignment (11–14):**
    * If the pitch is a Ball (outside the box), it assigns the "Shadow Zones" based on simple quadrants relative to the center of the plate.
    * **11:** High-Left | **12:** High-Right
    * **13:** Low-Left | **14:** Low-Right

* **Execution:**
    * The .apply() function runs this logic on every single row of the test dataframe, generating the final pitch_class and zone columns ready for submission CSV generation.

In [ ]:
# CELL 14: Assign pitch_class and zone based on model predictions

def assign_zone(row):
    x = row['guess_x']
    z = row['guess_z']
    top = row['sz_top']
    bot = row['sz_bot']

    # Strike?
    if abs(x) <= 0.83 and bot <= z <= top:
        pitch_class = 'strike'

        # 1–9 grid
        col = 0 if x < -0.27 else (2 if x > 0.27 else 1)
        row_z = 0 if z > (top - (top - bot)/3) else (
                2 if z < (bot + (top - bot)/3) else 1)
        zone = (row_z * 3) + col + 1

    else:
        pitch_class = 'ball'

        mid_z = (top + bot) / 2

        # 11–14 quadrants
        if x < 0 and z > mid_z:
            zone = 11
        elif x >= 0 and z > mid_z:
            zone = 12
        elif x < 0 and z <= mid_z:
            zone = 13
        else:
            zone = 14

    return pd.Series([pitch_class, zone])


test[['pitch_class', 'zone']] = test.apply(assign_zone, axis=1)
test[['file_name', 'pitch_class', 'zone']].head()

/tmp/ipykernel_10401/2913637839.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[['pitch_class', 'zone']] = test.apply(assign_zone, axis=1)
/tmp/ipykernel_10401/2913637839.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[['pitch_class', 'zone']] = test.apply(assign_zone, axis=1)


,file_name,pitch_class,zone
0,pitch2.mp4,ball,12
1,pitch6.mp4,ball,14
2,pitch7.mp4,strike,1
3,pitch9.mp4,strike,3
4,pitch10.mp4,strike,6


In [ ]:
# CELL 15: Save submission CSV

submission = test[['file_name', 'pitch_class', 'zone']].copy()

submission_path = "submission_neuralode_hybrid.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
submission.head()

Saved submission to: submission_neuralode_hybrid.csv


,file_name,pitch_class,zone
0,pitch2.mp4,ball,12
1,pitch6.mp4,ball,14
2,pitch7.mp4,strike,1
3,pitch9.mp4,strike,3
4,pitch10.mp4,strike,6


In [ ]:
# Cell 16: Ensure 'zone' is Float and Resave

# Force 'zone' to Float (matches Sample/GT)
submission['zone'] = submission['zone'].astype(float)

# Save
file_name = 'submission_neuralode_hybrid_float.csv'
submission.to_csv(file_name, index=False)

print(f"✅ Saved {file_name} with Float zones (e.g., 14.0).")
print(submission.head())

✅ Saved submission_neuralode_hybrid_float.csv with Float zones (e.g., 14.0).
     file_name pitch_class  zone
0   pitch2.mp4        ball  12.0
1   pitch6.mp4        ball  14.0
2   pitch7.mp4      strike   1.0
3   pitch9.mp4      strike   3.0
4  pitch10.mp4      strike   6.0
